# 02 — TensorRT Inference Sweep

For each trained checkpoint (`seed_1`, `seed_2`, `seed_42`), sweep all 5 precisions × 4 input bit-depths.

ONNX exports (base + QDQ) are assumed to already exist from `00_qdq_export.ipynb`.

Results saved under `runs/val_infer/<seed>/`, averaged at the end.

In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [15]:
import torch
import pandas as pd

from src.config import ExperimentConfig, with_overrides
from src.runner import run_experiment
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [16]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
ONNX_ROOT = PROJECT_ROOT / "onnx"
INPUT_BITS = [8, 4, 2, 1]
TRT_PRECISIONS = ["fp32", "fp16", "int8", "fp8", "int4"]

# Discover (bits, seed) pairs from checkpoints
experiments = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            seed = seed_dir.name
            seed_num = seed.split("_")[-1]
            experiments[(bits, seed)] = {
                "ckpt": str(seed_dir / "best.pth"),
                "seed_num": seed_num,
            }

def make_base(bits, seed, seed_num):
    return ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        trt_static_shape=True,
        trt_workspace_mb=2048,
        onnx_root=str(ONNX_ROOT / f"fp32_{bits}bit"),
        engine_root=str(PROJECT_ROOT / "engines" / f"fp32_{bits}bit" / seed),
        trt_onnx_prefix=f"resnet_{seed_num}",
        output_root=f"../runs/val_infer/fp32_{bits}bit/{seed}",
    )

print("Experiments found:")
for (bits, seed), info in experiments.items():
    print(f"  {bits}bit / {seed}: {info['ckpt']}")


Experiments found:
  8bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_1/best.pth
  8bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_2/best.pth
  8bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth
  4bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_1/best.pth
  4bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_2/best.pth
  4bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_42/best.pth
  2bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_1/best.pth
  2bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_2/best.pth
  2bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_42/best.pth
  1bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_1bit/seed_1/best

## FP32

In [17]:
fp32_records = []
print("TRT Precision: fp32")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp32", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    fp32_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: fp32
  SKIP 8bit / seed_1 (result exists)
  SKIP 8bit / seed_2 (result exists)
  SKIP 8bit / seed_42 (result exists)
  SKIP 4bit / seed_1 (result exists)
  SKIP 4bit / seed_2 (result exists)
  SKIP 4bit / seed_42 (result exists)
  SKIP 2bit / seed_1 (result exists)
  SKIP 2bit / seed_2 (result exists)
  SKIP 2bit / seed_42 (result exists)
  SKIP 1bit / seed_1 (result exists)
  SKIP 1bit / seed_2 (result exists)
  SKIP 1bit / seed_42 (result exists)


## FP16

In [18]:
fp16_records = []
print("TRT Precision: fp16")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp16", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    fp16_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: fp16
  SKIP 8bit / seed_1 (result exists)
  SKIP 8bit / seed_2 (result exists)
  SKIP 8bit / seed_42 (result exists)
  SKIP 4bit / seed_1 (result exists)
  SKIP 4bit / seed_2 (result exists)
  SKIP 4bit / seed_42 (result exists)
  SKIP 2bit / seed_1 (result exists)
  SKIP 2bit / seed_2 (result exists)
  SKIP 2bit / seed_42 (result exists)
  SKIP 1bit / seed_1 (result exists)
  SKIP 1bit / seed_2 (result exists)
  SKIP 1bit / seed_42 (result exists)


## INT8

In [19]:
int8_records = []
print("TRT Precision: int8")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="int8", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    int8_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: int8
  SKIP 8bit / seed_1 (result exists)
  SKIP 8bit / seed_2 (result exists)
  SKIP 8bit / seed_42 (result exists)

  4bit / seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1_int8_qdq.onnx
[runner] Step 2/3 — Building TRT engine (precision=int8) ...
[trt_builder] Building engine | precision=int8 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_4bit/seed_1/resnet18_tensorrt_int8_in4b_cuda_bs1.engine (11.7 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_4bit/seed_1/resnet18_tensorrt_int8_in4b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batche

## FP8

In [20]:
fp8_records = []
print("TRT Precision: fp8")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp8", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    fp8_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: fp8
  SKIP 8bit / seed_1 (result exists)
  SKIP 8bit / seed_2 (result exists)
  SKIP 8bit / seed_42 (result exists)

  4bit / seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1_fp8_qdq.onnx
[runner] Step 2/3 — Building TRT engine (precision=fp8) ...
[trt_builder] Building engine | precision=fp8 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_4bit/seed_1/resnet18_tensorrt_fp8_in4b_cuda_bs1.engine (11.8 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_4bit/seed_1/resnet18_tensorrt_fp8_in4b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — s

## INT4

In [21]:
int4_records = []
print("TRT Precision: int4")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="int4", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    int4_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: int4
  SKIP 8bit / seed_1 (result exists)
  SKIP 8bit / seed_2 (result exists)
  SKIP 8bit / seed_42 (result exists)

  4bit / seed_1:
[runner] Step 1/3 — ONNX exists, skipping: /home/pf4636/code/resnet/quantized_resnets/onnx/fp32_4bit/resnet_1_int4_qdq.onnx
[runner] Step 2/3 — Building TRT engine (precision=int4) ...
[trt_builder] Building engine | precision=int4 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_4bit/seed_1/resnet18_tensorrt_int4_in4b_cuda_bs1.engine (23.0 MB)
[runner] Step 3/3 — Running TRT inference ...
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_4bit/seed_1/resnet18_tensorrt_int4_in4b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 500 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batche

## All Results (per seed)

In [22]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"../runs/val_infer/fp32_{bits}bit/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)
df_trt = df[df["cfg.backend"] == "tensorrt"]

display_cols = [
    "seed", "input_bits_trained", "cfg.model_precision",
    "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.throughput_infer_sps",
]
df_trt[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision"],
    ascending=[False, True, True],
).reset_index(drop=True)


,seed,input_bits_trained,cfg.model_precision,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps
0,seed_1,8,fp16,98.936,100.000,0.488,2047.729
1,seed_1,8,fp32,98.936,100.000,0.965,1036.657
2,seed_1,8,fp8,98.723,100.000,0.660,1515.532
3,seed_1,8,int4,98.298,100.000,0.509,1965.832
4,seed_1,8,int8,98.936,100.000,0.665,1504.557
5,seed_2,8,fp16,98.085,100.000,0.483,2072.067
6,seed_2,8,fp32,98.085,100.000,0.957,1044.643
7,seed_2,8,fp8,98.298,100.000,0.683,1463.612
8,seed_2,8,int4,98.298,100.000,0.490,2038.849
9,seed_2,8,int8,98.298,100.000,0.719,1391.612


## Averaged Results (across seeds)

In [23]:
avg_df = df_trt.groupby(["cfg.backend", "cfg.model_precision", "input_bits_trained"]).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision", "input_bits_trained"],
    ascending=[True, False],
).reset_index(drop=True)
avg_df


,cfg.backend,cfg.model_precision,input_bits_trained,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,tensorrt,fp16,8,94.397,7.137,99.574,0.737,0.501,0.026,2001.118
1,tensorrt,fp16,4,94.610,6.402,99.645,0.614,0.503,0.010,1989.195
2,tensorrt,fp16,2,90.426,5.177,99.220,1.351,0.496,0.006,2017.660
3,tensorrt,fp16,1,86.028,5.809,97.801,2.152,0.486,0.014,2059.191
4,tensorrt,fp32,8,94.468,7.015,99.574,0.737,0.960,0.004,1041.982
5,tensorrt,fp32,4,94.610,6.402,99.645,0.614,0.952,0.020,1051.035
6,tensorrt,fp32,2,90.496,5.054,99.220,1.351,0.965,0.011,1036.294
7,tensorrt,fp32,1,86.028,5.809,97.801,2.152,0.952,0.015,1050.660
8,tensorrt,fp8,8,94.468,7.005,99.504,0.860,0.659,0.025,1519.019
9,tensorrt,fp8,4,94.184,6.961,99.574,0.737,0.663,0.024,1510.304


In [24]:
import json, numpy as np

bench_fp32 = PROJECT_ROOT / "results" / "latency_bench" / "tensorrt_fp32_in4b_cuda_bs1.json"
bench_int8 = PROJECT_ROOT / "results" / "latency_bench" / "tensorrt_int8_in8b_cuda_bs1.json"

with open(bench_fp32) as f:
    std_fp = np.std(json.load(f)["latencies_ms"], ddof=1)
with open(bench_int8) as f:
    std_int = np.std(json.load(f)["latencies_ms"], ddof=1)

bench_std_map = {"fp32": std_fp, "fp16": std_fp, "fp8": std_fp, "int8": std_int, "int4": std_int}

avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {bench_std_map[r['cfg.model_precision']]:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]


,cfg.model_precision,input_bits_trained,top1,top5,infer_ms
0,fp16,8,94.40 ± 7.14,99.57 ± 0.74,0.501 ± 0.136
1,fp16,4,94.61 ± 6.40,99.64 ± 0.61,0.503 ± 0.136
2,fp16,2,90.43 ± 5.18,99.22 ± 1.35,0.496 ± 0.136
3,fp16,1,86.03 ± 5.81,97.80 ± 2.15,0.486 ± 0.136
4,fp32,8,94.47 ± 7.01,99.57 ± 0.74,0.960 ± 0.136
5,fp32,4,94.61 ± 6.40,99.64 ± 0.61,0.952 ± 0.136
6,fp32,2,90.50 ± 5.05,99.22 ± 1.35,0.965 ± 0.136
7,fp32,1,86.03 ± 5.81,97.80 ± 2.15,0.952 ± 0.136
8,fp8,8,94.47 ± 7.00,99.50 ± 0.86,0.659 ± 0.136
9,fp8,4,94.18 ± 6.96,99.57 ± 0.74,0.663 ± 0.136


In [25]:
out_path = PROJECT_ROOT / "resultsv2/val_runs/tensorrt_avg_results.json"
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/resultsv2/val_runs/tensorrt_avg_results.json
